In [15]:
import requests
import re
import json
from urllib.parse import quote
from tqdm import tqdm
from bs4 import BeautifulSoup

In [16]:
URL_list = {
    'steam_simul_multi' : 'https://store.steampowered.com/search/results/?query&start=0&count=100&dynamic_data=&force_infinite=1&supportedlang=koreana&tags=599%2C3859&category1=998&filter=topsellers&ndl=1&snr=1_7_7_7000_7&infinite=1',
    'steam_simul_single' : 'https://store.steampowered.com/search/results/?query&start=0&count=100&dynamic_data=&force_infinite=1&filter=topsellers&supportedlang=koreana&tags=599%2C4182&category1=998&ndl=1&snr=1_7_7_7000_7&infinite=1',
    'steam_open_multi' : 'https://store.steampowered.com/search/results/?query&start=100&count=50&dynamic_data=&force_infinite=1&filter=topsellers&supportedlang=koreana&tags=1695%2C3859&category1=998&ndl=1&snr=1_7_7_7000_7&infinite=1',
    'steam_open_single' : 'https://store.steampowered.com/search/results/?query&start=0&count=100&dynamic_data=&force_infinite=1&filter=topsellers&supportedlang=koreana&tags=1695%2C4182&category1=998&ndl=1&snr=1_7_7_7000_7&infinite=1'
}

dict_steam_games = {}

header = {
    'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Whale/4.29.282.14 Safari/537.36'
    }

# 리뷰의 평가 기준 4개
reviews_tags = ['funny','summary','all','recent']

# 오픈월드, dlc 포함 x, 상위 100개를 불러옴
URL = URL_list['steam_simul_multi']

In [17]:
def games_info(link, result):
    '''2024년 이하의 게임 고유 번호, 타이틀명, 원가, 주소를 반환'''
    response = requests.get(link, headers=header)
    json_data = response.json()
    soup = BeautifulSoup(json_data['results_html'], 'html.parser')

    # 게임 고유 번호, 타이틀명, 원가, 주소 저장
    price = soup.select('a.search_result_row')
    game_cnt = 0

    for game in price:
        if int(game.select_one('div.search_released').text.strip()[-4:]) >= 2025:
            continue

        original_price = game.select_one('div.search_discount_and_price .discount_original_price')

        if original_price:
            # 원래 가격이 있을 경우
            price_value = original_price.text.replace('₩','').replace(',','').strip() 
            game_title =re.sub(r'[^\w\s_]','',game.select_one('span.title').text.replace(' ','_')) # 게임 타이틀을 저장
            game_code = game.get('data-ds-itemkey')[4:] # 게임 고유 번호를 저장
            game_link = game.get('href')
        else:
            # 원래 가격이 없으면 할인된 가격을 사용
            final_price = game.select_one(' div.search_discount_and_price .discount_final_price')
            price_value = final_price.text.replace('₩','').replace(',','').strip()
            game_title =re.sub(r'[^\w\s_]','',game.select_one('span.title').text.replace(' ','_'))
            game_code = game.get('data-ds-itemkey')[4:]
            game_link = game.get('href')

            if price_value == 'Free':
                price_value = 0

        result[game_title] = {}
        sub_result = result[game_title]
        sub_result['code'] = game_code
        sub_result['price'] = int(price_value)
        sub_result['link'] = game_link
    
        game_cnt += 1
        if game_cnt == 50:
            break

    return result

def games_time_avg(code):
    '''재밌는, 유용한, 최근의, 모든 평가 각각의 기준별로 상위 100개의 평가를 한 플레이어의 시간 합, 평균을 구함'''
    total_time = []

    for tag in reviews_tags:
        cursor = '*'
        for _ in range(5):
            URL = f'https://store.steampowered.com/appreviews/{code}?use_review_quality=1&cursor={cursor}&day_range=30&start_date=-1&end_date=-1&date_range_type=all&filter={tag}&language=english&l=english&review_type=all&purchase_type=all&playtime_filter_min=0&playtime_filter_max=0&playtime_type=all&filter_offtopic_activity=1&summary_num_positive_reviews=292163&summary_num_reviews=310508'
            response = requests.get(URL, headers=header)
            reviews = response.json()
            cursor = quote(reviews['cursor'])
            soup = BeautifulSoup(reviews['html'],'html.parser')
        
            total_time += [float(time.text.strip().split(' ')[0].replace(',','')) for time in soup.select('div.hours')]
    try:
        time_avg = round(sum(total_time) / len(total_time), 2)
    # 평가를 하지 못하는 확장판, DLC 등의 에러 처리
    except ZeroDivisionError:
        time_avg = 'Error'
    return time_avg

def find_key_by_value(d=URL_list, target_value=URL):
    for key, value in d.items():
        if value == target_value:
            return key  # 첫 번째로 찾은 key 반환
    return None  # 값이 없을 경우 None 반환

def achievement_info(value):
    '''게임 별 업적의 정보를 크롤링'''
    url = f'https://steamcommunity.com/stats/{value['code']}/achievements'
    response = requests.get(url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')
    try:
        total_ach = int(soup.select_one('div.maincontent span').text) # 총 업적 개수

        value['total_ach'] = total_ach
        value['ach_list'] = {}
        ach_list = value['ach_list']
        
        for ach in soup.select('div.maincontent div.achieveRow'):
            ach_title = ach.select_one('div.achieveTxt').text.strip().split('\n')[0] # 0번 업적 이름, 1번 업적 내용
            ach_percent = ach.select_one('div.achievePercent').text # 해당 업적 달성률
            ach_percent = float(ach_percent[:-1])
            ach_list[ach_title] = ach_percent

        
    # 업적이 없는 게임일 경우
    except:
        value['total_ach'] = 0

    return value

def ach_ratio_calculation(data) :
    '''게임 업적 비율의 전반적인 계산 (합계, 평균)'''
    for k, v in data.items():
        ahc_ratio_list = []
        try:
            ach_ratio_sum = 0
            for sub_k, sub_v in v['ach_list'].items():
                ach_ratio_sum += sub_v
                ahc_ratio_list.append(sub_v)
                v['ach_ratio_sum'] = round(ach_ratio_sum, 2) # 업적의 비율의 합계
                v['ach_ratio_avg'] = round(v['ach_ratio_sum'] / v['total_ach'], 2) # 총 업적의 평균
                v['ach_ratio_list'] = ahc_ratio_list # 업적 비율 수치만 리스트로 저장
        except:
            ach_ratio_sum = 0
            v['ach_ratio_sum'] = round(ach_ratio_sum, 2) # 업적의 비율의 합계
            v['ach_ratio_avg'] = 0 # 총 업적의 평균
            v['ach_ratio_list'] = ahc_ratio_list # 업적 비율 수치만 리스트로 저장
    
    return data 

In [18]:
# 게임의 기본적인 정보 크롤링
dict_steam_games = games_info(URL, dict_steam_games)  

# 해당 게임의 평가 기준 총 플레이 시간의 평균을 반환
for key, value in tqdm(dict_steam_games.items(), desc="Processing Games"):
    value['time_avg'] = games_time_avg(value['code'])

Processing Games: 100%|██████████| 50/50 [07:20<00:00,  8.81s/it]


In [19]:
# 평가가 없는 (플레이 시간이 에러인) 게임 제외, 확장판은 리뷰 및 평가를 제공하지 않음
dict_filtered = {k: v for k, v in dict_steam_games.items() if v['time_avg']!='Error'}

for k,v in tqdm(dict_filtered.items(), desc="Processing"):
    dict_filtered[k] = achievement_info(v)

dict_filtered = ach_ratio_calculation(dict_filtered)
dict_filtered

Processing: 100%|██████████| 50/50 [00:19<00:00,  2.57it/s]


{'PUBG_BATTLEGROUNDS': {'code': '578080',
  'price': 0,
  'link': 'https://store.steampowered.com/app/578080/PUBG_BATTLEGROUNDS/?snr=1_7_7_7000_150_1',
  'time_avg': 499.34,
  'total_ach': 37,
  'ach_list': {'Blood on My Hands': 65.2,
   'Health Junkie': 54.5,
   'Guardian Angel': 53.7,
   'Devil Inside Me': 50.4,
   'Killing Spree': 50.0,
   'Pacifist': 49.0,
   'You Complete Me': 47.9,
   'Marksman Novice': 42.2,
   'Fantastic Four': 41.7,
   'Top 10': 36.2,
   'Marksman Expert': 31.7,
   "Now You See Me, Now You Don't": 31.4,
   'First Blood': 30.9,
   'CQB Novice': 30.2,
   'Collateral Damage': 28.6,
   'Airborne': 26.5,
   'Dynamic Duo': 22.0,
   'Marksman Master': 21.2,
   'First Come, First Served': 19.6,
   'Last Survivor': 19.2,
   'Trigonometry Involved': 19.1,
   'CQB Expert': 17.5,
   "Okay, Now I'm Ready": 17.4,
   'Nade King Novice': 14.3,
   'Agent 48': 13.9,
   'Winner Winner Chicken Dinner!': 12.5,
   'Cruising with the Enemy': 12.4,
   'Long and Winding Road': 10.0,
 

In [21]:
# json 형식으로 저장
with open('data/' + find_key_by_value(URL_list, URL) + '.json', "w", encoding="utf-8") as json_file:
    json.dump(dict_filtered, json_file, ensure_ascii=False, indent=4)